# Technology And Designer Resume Paraphrasing

This notebook filters the two resume CSVs by keyword, randomly samples matching resumes, asks the OpenAI API for 4 faithful paraphrases of each resume, and saves the results to JSON.

Configured jobs:
- `Resume_INFORMATION-TECHNOLOGY.csv` filtered by resumes containing both `software` and `engineer`, saved to `it_resume_paraphrases_sample_10.json`
- `Resume_DESIGNER.csv` filtered by `graphic designer`, saved to `designer_resume_paraphrases_sample_10.json`

Requirements:
- `OPENAI_API_KEY` must be available in the environment.
- The CSV files must be the real datasets, not Git LFS pointer files.


In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

MODEL = "gpt-5.4-mini"
SAMPLE_SIZE = 10
PARAPHRASES_PER_RESUME = 4
RANDOM_SEED = 42
MAX_RETRIES = 3
RESUME_TEXT_COLUMN = "Resume_str"


def resolve_data_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "datasets/Tech_Designer",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "Resume_INFORMATION-TECHNOLOGY.csv").exists() and (candidate / "Resume_DESIGNER.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Tech_Designer dataset directory from the current working directory.")


DATA_DIR = resolve_data_dir()
JOB_CONFIGS = [
    {
        "label": "it",
        "dataset_path": DATA_DIR / "Resume_INFORMATION-TECHNOLOGY.csv",
        "match_terms": ["software", "engineer"],
        "output_path": DATA_DIR / "it_resume_paraphrases_sample_10.json",
    },
    {
        "label": "designer",
        "dataset_path": DATA_DIR / "Resume_DESIGNER.csv",
        "match_terms": ["graphic designer"],
        "output_path": DATA_DIR / "designer_resume_paraphrases_sample_10.json",
    },
]

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

for job in JOB_CONFIGS:
    print(f"Dataset path: {job['dataset_path']}")
    print(f"Match terms:  {job['match_terms']}")
    print(f"Output path:  {job['output_path']}")
    print(f"Model:        {MODEL}")
    print()


In [ ]:
def assert_not_lfs_pointer(path: Path) -> None:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        first_line = handle.readline().strip()

    if first_line == "version https://git-lfs.github.com/spec/v1":
        raise RuntimeError(
            "The dataset file is currently a Git LFS pointer, not the real CSV. "
            "Replace it with the actual dataset contents before running this notebook."
        )


def load_keyword_matches(path: Path, match_terms: list[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    assert_not_lfs_pointer(path)
    df = pd.read_csv(path)

    required_columns = {"ID", RESUME_TEXT_COLUMN, "Category"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

    cleaned = df[["ID", "Category", RESUME_TEXT_COLUMN]].copy()
    cleaned[RESUME_TEXT_COLUMN] = cleaned[RESUME_TEXT_COLUMN].fillna("").astype(str)
    cleaned = cleaned[cleaned[RESUME_TEXT_COLUMN].str.strip() != ""].reset_index(drop=True)

    mask = pd.Series(True, index=cleaned.index)
    for term in match_terms:
        mask &= cleaned[RESUME_TEXT_COLUMN].str.contains(term, case=False, na=False, regex=False)

    return cleaned[mask].reset_index(drop=True)


def load_keyword_sample(path: Path, match_terms: list[str]) -> pd.DataFrame:
    matches = load_keyword_matches(path, match_terms)
    available = len(matches)
    match_description = " and ".join(repr(term) for term in match_terms)

    if available == 0:
        raise ValueError(f"No resumes in {path.name} contain {match_description}.")

    sample_size = min(SAMPLE_SIZE, available)
    if sample_size < SAMPLE_SIZE:
        print(
            f"Warning: {path.name} only has {available} resumes containing {match_description}. "
            f"Sampling all {available} matches instead of {SAMPLE_SIZE}."
        )

    return matches.sample(n=sample_size, random_state=RANDOM_SEED).reset_index(drop=True)


for job in JOB_CONFIGS:
    matches = load_keyword_matches(job["dataset_path"], job["match_terms"])
    print(f"{job['dataset_path'].name}: {len(matches)} matches for {job['match_terms']}")


In [ ]:
PARAPHRASE_KEYS = [f"paraphrase_{index}" for index in range(1, PARAPHRASES_PER_RESUME + 1)]
PARAPHRASE_SCHEMA = {
    "type": "object",
    "properties": {key: {"type": "string"} for key in PARAPHRASE_KEYS},
    "required": PARAPHRASE_KEYS,
    "additionalProperties": False,
}

SYSTEM_PROMPT = """You are a meticulous resume paraphrasing assistant.
Your job is to rewrite a resume into four distinct paraphrases.

Hard requirements:
- Preserve every piece of meaning from the original text.
- Do not add, remove, summarize, infer, normalize, or embellish anything.
- Keep all facts, numbers, dates, contact details, technologies, tools, employers, schools, degrees, locations, and achievements.
- Only change grammar, wording, phrasing, and local sentence structure.
- Keep the output in English.
- Return JSON only.
"""


def paraphrase_resume(resume_text: str) -> dict:
    user_prompt = f"""Generate exactly {PARAPHRASES_PER_RESUME} faithful paraphrases of the resume below.

The paraphrases must differ in wording and grammar, but they must preserve all meaning and all details.

Resume:
<<RESUME_START>>
{resume_text}
<<RESUME_END>>
"""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "resume_paraphrases",
                        "strict": True,
                        "schema": PARAPHRASE_SCHEMA,
                    }
                },
            )
            return json.loads(response.output_text)
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(attempt * 2)

    raise RuntimeError("Unexpected retry flow.")


def run_job(job: dict) -> list[dict]:
    sample_df = load_keyword_sample(job["dataset_path"], job["match_terms"])
    results = []

    print(f"Running {job['label']} batch with {len(sample_df)} sampled resumes...")
    for index, row in sample_df.iterrows():
        paraphrases = paraphrase_resume(row[RESUME_TEXT_COLUMN])
        result = {
            "resume_id": int(row["ID"]),
            "category": row["Category"],
            "matched_terms": job["match_terms"],
            "original": row[RESUME_TEXT_COLUMN],
        }
        result.update({key: paraphrases[key] for key in PARAPHRASE_KEYS})
        results.append(result)
        print(f"Completed {index + 1}/{len(sample_df)} resumes for {job['label']}")

    job["output_path"].parent.mkdir(parents=True, exist_ok=True)
    job["output_path"].write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(results)} records to {job['output_path']}")
    return results


In [ ]:
results_by_job = {}

for job in JOB_CONFIGS:
    print()
    results_by_job[job["label"]] = run_job(job)

results_by_job.keys()
